In [8]:
import pandas as pd

df = pd.read_csv("books.csv")

# Clean and fill missing values for consistency
df = df.fillna("Unknown")

def textual_representation(row):
    return f"""Title: {row['title']}
Authors: {row['authors']}
Description: {row['description']}
Categories: {row['categories']}
Publishing Year: {row['published_year']}
Average Rating: {row['average_rating']}
Number of Pages: {row['num_pages']}"""

df["textual_representation"] = df.apply(textual_representation, axis=1)


In [9]:
from langchain.schema import Document

docs = [
    Document(
        page_content=row["textual_representation"],
        metadata={
            "title": row["title"],
            "categories": row["categories"],
            "num_pages": row["num_pages"],
            "published_year": row["published_year"],
        }
    )
    for _, row in df.iterrows()
]


In [10]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
split_docs = splitter.split_documents(docs)


In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(split_docs, embedding)
